In [1]:
import itertools
import math
from sympy import *
import time

In this notebook, we look for solutions to $p(r,s)/D(r,s)=q(r,s)/D(r,s)=0$, where $p(r,s)=r^a+s^tr^b-s^ur^c-s^vr^d$, $q(r,s)=r^e+s^xr^f-s^yr^g-s^zr^h$, $D=\gcd(p,q)$, $r,s\in \mathbb{Q}$, $r,s>1$, and $\log_r(s)\notin \mathbb{Q}$, as these correspond to simultaneously occurring pairs of geometric families of repeated sums in our two-dimensional geometric progressions. The cases where $D(r,s)=0$ are accounted for in the 'GCD Check' notebook. We collect all pairs $(r,s)$ for which these collisions occur, and check this list of exceptions separately for small sumsets, as seen in the 'Classification' and 'Uniqueness' notebooks.

The good_octuple function below is the same as in the 'GCD check' notebook, which prevents redundancy and degeneracy in solutions to $p(r,s)=q(r,s)=0$.

In [2]:
def good_octuple(a,b,c,d,e,f,g,h,t,u,v,x,y,z):
    good = True
    if not (min(a,b,c,d)==min(e,f,g,h)==0):
        good = False
    if (u==0 and a==c) or (v==0 and a==d) or (t==u and b==c) or (t==v and b==d):
        good = False
    if (y==0 and e==g) or (z==0 and e==h) or (x==y and f==g) or (x==z and f==h):
        good = False
    if (t==0 and a<b) or (u==v and c<d) or (x==0 and e<f) or (y==z and g<h) or (u==0 and t==v and c>=a) or (y==0 and x==z and e<=g):
        good = False

    if (t==x==0 and {a,b}=={e,f}) or (u==v==y==z and {c,d}=={g,h}):
        good = False
    if ((0,t) == (u,v) == (0,x) == (y,z)) and (((a,b) == (e,f) and (c,d) == (g, h)) or ((a,b) == (g,h) and (c,d) == (e,f))):
        good = False

    if ((t,u,v)==(x,y,z) and a-e==b-f==c-g==d-h):
        good = False

    return good

The following function takes in $(a,b,c,d,e,f,g,h,t,u,v,x,y,z)$ and returns $p/D,q/D$.

In [3]:
def factor_out_gcd(a,b,c,d,e,f,g,h,t,u,v,x,y,z):
    r, s = symbols('r s')
    p = r**(a)+s**t*r**(b)-s**u*r**(c)-s**v*r**(d)
    q = r**(e)+s**x*r**(f)-s**y*r**(g)-s**z*r**(h)
   
    G = gcd(p,q) 

    p = (p/G).cancel()
    q = (q/G).cancel()
           
    return p,q

The function below collects rational roots that are greater than $1$. 

In [4]:
def get_rat_roots(p,r):
    rats = set([])
    g = factor_list(p)
    for x in g[1]:
                y = x[0]
                if degree(y)==1:
                    a = y.coeff(r)
                    b = y.coeff(r,0)
                    z = Rational(-b,a)
                    if z>1:
                        rats.add(z)
    return rats

Below is the set $T_2$ from the paper holding the possible values of $(t,u,v)$ and $(x,y,z)$. The first four elements form $T_1$.

In [5]:
ExponentTuples= [(0,1,1),(0,0,1),(1,1,1),(1,0,1),(0,2,2),(0,0,2),(2,2,2),(2,0,2),(1,1,2),(2,1,1),(1,2,2),(2,1,2),(0,1,2),(1,0,2)]

We create dictionaries to store two and three-row collisions, respectively.

In [6]:
Two_row_Collision_Dictionary = {}
for i in range(4):
    for j in range(i,4):
        (t,u,v) = ExponentTuples[i]
        (x,y,z) = ExponentTuples[j]
        Two_row_Collision_Dictionary[(t,u,v,x,y,z)] = []

In [7]:
Three_row_Collision_Dictionary = {}
for i in range(14):
    for j in range(i,14):
        (t,u,v) = ExponentTuples[i]
        (x,y,z) = ExponentTuples[j]
        Three_row_Collision_Dictionary[(t,u,v,x,y,z)] = []

The following two functions implement CollisionCheck as described in the paper.

In [7]:
def collision_type_check(t,u,v,x,y,z,l):
  start = time.time()
  collisions = []
  L = range(l)
  r, s = symbols('r s')
  for (a,b,c,d,e,f,g,h) in itertools.product(L,L,L,L,L,L,L,L):
      if good_octuple(a,b,c,d,e,f,g,h,t,u,v,x,y,z):
          p,q = factor_out_gcd(a,b,c,d,e,f,g,h,t,u,v,x,y,z)
          res1 = resultant(p,q,s)
          res2 = resultant(p,q,r)
          if res1!=0 and res2!=0:
                    r_roots = get_rat_roots(res1,r)
                    s_roots = get_rat_roots(res2,s)
                    for r1 in r_roots:
                        for s1 in s_roots:
                            p1 = p.subs({r:r1,s:s1})
                            q1 = q.subs({r:r1,s:s1})
                            if r1 != s1 and r1**2 != s1 and p1 == q1 == 0:
                                collisions.append((r1,s1,a,b,c,d,e,f,g,h))
  end = time.time()
  print(round(end-start,2))        
  return collisions

In [8]:
def Collision_Check(rows,l):
    start_all = time.time()
    done = 0
    if rows == 2:
        ETs = Two_row_Collision_Dictionary.keys()
    if rows == 3:
        ETs = Three_row_Collision_Dictionary.keys()
    for (t,u,v,x,y,z) in ETs:
            collisions = collision_type_check(t,u,v,x,y,z,l)
            if rows == 2:  
                Two_row_Collision_Dictionary[(t,u,v,x,y,z)] = collisions
            if rows == 3:
                Three_row_Collision_Dictionary[(t,u,v,x,y,z)] = collisions
            done = done+1
            print(done)
    end_all = time.time()
    print('Total time elapsed (in sec.):', round(end_all-start_all,2))

We execute Collision_Check(3,4) for three-row collisions, and Collision_Check(2,8) for two-row collisions, and extract the set of occurring $(r,s)$ pairs to store as lists of exceptions to be checked separately in the 'Classification' and 'Uniqueness' notebooks.

In [10]:
Collision_Check(3,4)

61.37
1
81.32
2
84.77
3
83.53
4
113.43
5
101.75
6
117.73
7
91.45
8
225.81
9
145.02
10
171.83
11
234.68
12
155.01
13
217.52
14
71.16
15
92.41
16
68.15
17
93.09
18
94.9
19
121.31
20
91.59
21
217.84
22
163.66
23
177.22
24
237.78
25
148.95
26
215.56
27
78.78
28
73.1
29
111.16
30
123.85
31
116.97
32
95.63
33
231.33
34
156.82
35
179.24
36
232.35
37
183.54
38
238.31
39
43.67
40
78.03
41
85.16
42
88.29
43
52.62
44
164.99
45
121.95
46
131.45
47
162.95
48
129.2
49
164.11
50
65.02
51
87.96
52
91.86
53
77.41
54
231.79
55
181.87
56
167.44
57
253.97
58
161.84
59
246.34
60
89.75
61
117.92
62
86.02
63
248.67
64
195.6
65
210.61
66
288.74
67
173.45
68
254.62
69
95.99
70
90.92
71
266.54
72
208.76
73
189.62
74
275.48
75
220.16
76
297.5
77
50.11
78
202.65
79
156.47
80
168.74
81
207.38
82
166.14
83
206.11
84
475.88
85
376.45
86
402.41
87
539.88
88
395.17
89
524.04
90
228.81
91
302.3
92
400.45
93
296.15
94
387.79
95
264.03
96
414.55
97
318.26
98
420.09
99
490.46
100
424.34
101
564.51
102
252.52
103
393.29
10

In [11]:
Three_row_Exceptions = set([])
for v in Three_row_Collision_Dictionary.values():
    for (r,s,a,b,c,d,e,f,g,h) in v:
        Three_row_Exceptions.add((r,s))
Three_row_Exceptions

{(5/4, 6/5),
 (5/4, 3/2),
 (5/4, 8/5),
 (5/4, 2),
 (5/4, 5/2),
 (4/3, 9/8),
 (4/3, 8/7),
 (4/3, 7/6),
 (4/3, 3/2),
 (4/3, 2),
 (4/3, 8/3),
 (3/2, 9/8),
 (3/2, 6/5),
 (3/2, 5/4),
 (3/2, 4/3),
 (3/2, 2),
 (3/2, 8/3),
 (3/2, 3),
 (3/2, 4),
 (3/2, 9/2),
 (3/2, 6),
 (5/3, 7/5),
 (5/3, 9/5),
 (5/3, 7/3),
 (5/3, 3),
 (5/3, 5),
 (9/5, 5/3),
 (9/5, 3),
 (9/5, 27/5),
 (2, 9/8),
 (2, 8/7),
 (2, 5/4),
 (2, 4/3),
 (2, 3/2),
 (2, 8/5),
 (2, 7/4),
 (2, 9/4),
 (2, 5/2),
 (2, 8/3),
 (2, 3),
 (2, 7/2),
 (2, 9/2),
 (2, 5),
 (2, 6),
 (2, 7),
 (5/2, 5/4),
 (5/2, 8/5),
 (5/2, 2),
 (5/2, 4),
 (5/2, 5),
 (5/2, 10),
 (3, 4/3),
 (3, 3/2),
 (3, 5/3),
 (3, 9/5),
 (3, 2),
 (3, 4),
 (3, 9/2),
 (3, 5),
 (3, 27/5),
 (3, 6),
 (3, 15),
 (4, 3/2),
 (4, 8/5),
 (4, 7/4),
 (4, 5/2),
 (4, 8/3),
 (4, 6),
 (4, 7),
 (4, 10),
 (5, 5/3),
 (5, 9/5),
 (5, 3),
 (5, 9),
 (5, 15)}

In [12]:
len(Three_row_Exceptions)

75

In [13]:
Three_row_permanent = {(5/4, 6/5),
 (5/4, 3/2),
 (5/4, 8/5),
 (5/4, 2),
 (5/4, 5/2),
 (4/3, 9/8),
 (4/3, 8/7),
 (4/3, 7/6),
 (4/3, 3/2),
 (4/3, 2),
 (4/3, 8/3),
 (3/2, 9/8),
 (3/2, 6/5),
 (3/2, 5/4),
 (3/2, 4/3),
 (3/2, 2),
 (3/2, 8/3),
 (3/2, 3),
 (3/2, 4),
 (3/2, 9/2),
 (3/2, 6),
 (5/3, 7/5),
 (5/3, 9/5),
 (5/3, 7/3),
 (5/3, 3),
 (5/3, 5),
 (9/5, 5/3),
 (9/5, 3),
 (9/5, 27/5),
 (2, 9/8),
 (2, 8/7),
 (2, 5/4),
 (2, 4/3),
 (2, 3/2),
 (2, 8/5),
 (2, 7/4),
 (2, 9/4),
 (2, 5/2),
 (2, 8/3),
 (2, 3),
 (2, 7/2),
 (2, 9/2),
 (2, 5),
 (2, 6),
 (2, 7),
 (5/2, 5/4),
 (5/2, 8/5),
 (5/2, 2),
 (5/2, 4),
 (5/2, 5),
 (5/2, 10),
 (3, 4/3),
 (3, 3/2),
 (3, 5/3),
 (3, 9/5),
 (3, 2),
 (3, 4),
 (3, 9/2),
 (3, 5),
 (3, 27/5),
 (3, 6),
 (3, 15),
 (4, 3/2),
 (4, 8/5),
 (4, 7/4),
 (4, 5/2),
 (4, 8/3),
 (4, 6),
 (4, 7),
 (4, 10),
 (5, 5/3),
 (5, 9/5),
 (5, 3),
 (5, 9),
 (5, 15)}

In [9]:
Collision_Check(2,8)

2907.78
1
4731.77
2
5003.98
3
4747.22
4
6645.45
5
7841.13
6
6087.5
7
6280.4
8
6340.04
9
5099.55
10
Total time elapsed (in sec.): 55684.91


Three-row runtime: 21100 seconds

Two-row runtime: 55700 seconds

In [10]:
Two_row_Exceptions = set([])
for v in Two_row_Collision_Dictionary.values():
    for (r,s,a,b,c,d,e,f,g,h) in v:
        Two_row_Exceptions.add((r,s))
Two_row_Exceptions

{(3/2, 9/8),
 (3/2, 4/3),
 (3/2, 27/16),
 (3/2, 2),
 (3/2, 81/32),
 (3/2, 3),
 (3/2, 243/64),
 (3/2, 9/2),
 (3/2, 729/128),
 (3/2, 27/4),
 (3/2, 81/8),
 (3/2, 243/16),
 (3/2, 729/32),
 (2, 129/128),
 (2, 128/127),
 (2, 65/64),
 (2, 64/63),
 (2, 33/32),
 (2, 32/31),
 (2, 17/16),
 (2, 16/15),
 (2, 9/8),
 (2, 8/7),
 (2, 5/4),
 (2, 4/3),
 (2, 11/8),
 (2, 16/11),
 (2, 3/2),
 (2, 8/5),
 (2, 7/4),
 (2, 16/9),
 (2, 15/8),
 (2, 32/17),
 (2, 31/16),
 (2, 64/33),
 (2, 63/32),
 (2, 128/65),
 (2, 127/64),
 (2, 129/64),
 (2, 65/32),
 (2, 128/63),
 (2, 33/16),
 (2, 64/31),
 (2, 17/8),
 (2, 32/15),
 (2, 9/4),
 (2, 16/7),
 (2, 5/2),
 (2, 8/3),
 (2, 11/4),
 (2, 32/11),
 (2, 3),
 (2, 16/5),
 (2, 7/2),
 (2, 32/9),
 (2, 15/4),
 (2, 64/17),
 (2, 31/8),
 (2, 128/33),
 (2, 63/16),
 (2, 127/32),
 (2, 129/32),
 (2, 65/16),
 (2, 33/8),
 (2, 128/31),
 (2, 17/4),
 (2, 64/15),
 (2, 9/2),
 (2, 32/7),
 (2, 5),
 (2, 16/3),
 (2, 11/2),
 (2, 64/11),
 (2, 6),
 (2, 32/5),
 (2, 7),
 (2, 64/9),
 (2, 15/2),
 (2, 128/17),
 (2

In [11]:
len(Two_row_Exceptions)

155

In [12]:
Two_row_permanent = {(3/2, 9/8),
 (3/2, 4/3),
 (3/2, 27/16),
 (3/2, 2),
 (3/2, 81/32),
 (3/2, 3),
 (3/2, 243/64),
 (3/2, 9/2),
 (3/2, 729/128),
 (3/2, 27/4),
 (3/2, 81/8),
 (3/2, 243/16),
 (3/2, 729/32),
 (2, 129/128),
 (2, 128/127),
 (2, 65/64),
 (2, 64/63),
 (2, 33/32),
 (2, 32/31),
 (2, 17/16),
 (2, 16/15),
 (2, 9/8),
 (2, 8/7),
 (2, 5/4),
 (2, 4/3),
 (2, 11/8),
 (2, 16/11),
 (2, 3/2),
 (2, 8/5),
 (2, 7/4),
 (2, 16/9),
 (2, 15/8),
 (2, 32/17),
 (2, 31/16),
 (2, 64/33),
 (2, 63/32),
 (2, 128/65),
 (2, 127/64),
 (2, 129/64),
 (2, 65/32),
 (2, 128/63),
 (2, 33/16),
 (2, 64/31),
 (2, 17/8),
 (2, 32/15),
 (2, 9/4),
 (2, 16/7),
 (2, 5/2),
 (2, 8/3),
 (2, 11/4),
 (2, 32/11),
 (2, 3),
 (2, 16/5),
 (2, 7/2),
 (2, 32/9),
 (2, 15/4),
 (2, 64/17),
 (2, 31/8),
 (2, 128/33),
 (2, 63/16),
 (2, 127/32),
 (2, 129/32),
 (2, 65/16),
 (2, 33/8),
 (2, 128/31),
 (2, 17/4),
 (2, 64/15),
 (2, 9/2),
 (2, 32/7),
 (2, 5),
 (2, 16/3),
 (2, 11/2),
 (2, 64/11),
 (2, 6),
 (2, 32/5),
 (2, 7),
 (2, 64/9),
 (2, 15/2),
 (2, 128/17),
 (2, 31/4),
 (2, 63/8),
 (2, 127/16),
 (2, 129/16),
 (2, 65/8),
 (2, 33/4),
 (2, 17/2),
 (2, 128/15),
 (2, 9),
 (2, 64/7),
 (2, 10),
 (2, 32/3),
 (2, 11),
 (2, 12),
 (2, 64/5),
 (2, 14),
 (2, 128/9),
 (2, 15),
 (2, 31/2),
 (2, 63/4),
 (2, 127/8),
 (2, 129/8),
 (2, 65/4),
 (2, 33/2),
 (2, 17),
 (2, 18),
 (2, 128/7),
 (2, 20),
 (2, 64/3),
 (2, 22),
 (2, 24),
 (2, 128/5),
 (2, 28),
 (2, 30),
 (2, 31),
 (2, 63/2),
 (2, 127/4),
 (2, 129/4),
 (2, 65/2),
 (2, 33),
 (2, 34),
 (2, 36),
 (2, 40),
 (2, 128/3),
 (2, 44),
 (2, 48),
 (2, 56),
 (2, 60),
 (2, 62),
 (2, 63),
 (2, 127/2),
 (2, 129/2),
 (2, 65),
 (2, 66),
 (2, 68),
 (2, 72),
 (2, 80),
 (2, 96),
 (2, 112),
 (2, 120),
 (2, 124),
 (2, 126),
 (2, 127),
 (3, 5/3),
 (3, 9/5),
 (3, 5),
 (3, 27/5),
 (3, 15),
 (3, 81/5),
 (3, 45),
 (3, 243/5),
 (3, 135),
 (3, 729/5),
 (3, 405),
 (3, 2187/5),
 (3, 1215)}